# Вариант 1

In [1]:
import pandas as pd
import numpy as np

# В JupyterLab Web (Pyodide) нет системной базы таймзон, загружаем tzdata
try:
    import pyodide_js
    await pyodide_js.loadPackage("tzdata")
except (ImportError, AttributeError):
    try:
        import micropip
        await micropip.install("tzdata")
    except (ImportError, AttributeError):
        pass

# Практическая работа 4
### Временные ряды и пользовательские сессии

**Цель:** Научиться работать с датами и временем в pandas, выполнять агрегирование по временным интервалам и использовать оконные функции для анализа активности пользователей.

### Исходные данные

Файлы заказов интернет-магазина за три месяца:
- `data/orders_2024_01.csv` — январь 2024
- `data/orders_2024_02.csv` — февраль 2024
- `data/orders_2024_03.csv` — март 2024

Столбцы: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`

### Подготовка: Загрузка данных и парсинг datetime

In [2]:
# Загружаем все три файла и объединяем в один DataFrame
orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders = pd.concat(
    [pd.read_csv(f) for f in orders_files],
    ignore_index=True
)

# Преобразуем строку в тип datetime
df_orders["order_datetime"] = pd.to_datetime(df_orders["order_datetime"])

print("Форма таблицы:", df_orders.shape)
print("Тип order_datetime:", df_orders["order_datetime"].dtype)
df_orders.head()

Форма таблицы: (15000, 7)
Тип order_datetime: datetime64[us]


   order_id  user_id      order_datetime  product_id     price  quantity    status
0         1     1044 2024-01-03 18:39:26         201  14038.15         1  refunded
1         2     1079 2024-01-03 17:45:43         215   9172.60         1      paid
2         3     1041 2024-01-06 22:46:38         208   9035.26         2      paid
3         4     1083 2024-01-08 21:03:51         203  15234.33         3      paid
4         5     1068 2024-01-03 23:26:53         212   4134.38         2      paid

In [3]:
# Выделяем дату, час и день недели
df_orders["order_date"]    = df_orders["order_datetime"].dt.date
df_orders["order_hour"]    = df_orders["order_datetime"].dt.hour
df_orders["order_weekday"] = df_orders["order_datetime"].dt.day_name()

# Назначаем часовой пояс UTC, затем переводим в московское время
df_orders["order_datetime_utc"] = df_orders["order_datetime"].dt.tz_localize("UTC")
try:
    df_orders["order_datetime_msk"] = df_orders["order_datetime_utc"].dt.tz_convert("Europe/Moscow")
except Exception:
    # Запасной вариант для сред без IANA tzdata (смещение Москвы UTC+3)
    df_orders["order_datetime_msk"] = df_orders["order_datetime_utc"].dt.tz_convert("+03:00")

df_orders[["order_datetime", "order_datetime_utc", "order_datetime_msk"]].head()

        order_datetime        order_datetime_utc        order_datetime_msk
0  2024-01-03 18:39:26  2024-01-03 18:39:26+00:00  2024-01-03 21:39:26+03:00
1  2024-01-03 17:45:43  2024-01-03 17:45:43+00:00  2024-01-03 20:45:43+03:00
2  2024-01-06 22:46:38  2024-01-06 22:46:38+00:00  2024-01-07 01:46:38+03:00
3  2024-01-08 21:03:51  2024-01-08 21:03:51+00:00  2024-01-08 00:03:51+03:00
4  2024-01-03 23:26:53  2024-01-03 23:26:53+00:00  2024-01-04 02:26:53+03:00

### Задание 1. Построить дневную агрегированную таблицу по заказам: количество заказов и сумма выручки

In [4]:
# Устанавливаем московское время как индекс и сортируем
orders_ts = df_orders.set_index("order_datetime_msk").sort_index()

# Добавляем столбец выручки
orders_ts["revenue"] = orders_ts["price"] * orders_ts["quantity"]

# Агрегация по дням: количество заказов и суммарная выручка
daily_orders = orders_ts.resample("D").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum")
)

print("Дневная таблица заказов (первые 10 дней):")
daily_orders.head(10)

Дневная таблица заказов (первые 10 дней):


                           orders_count  revenue_sum
order_datetime_msk
2024-01-01 00:00:00+03:00           141   3888229.53
2024-01-02 00:00:00+03:00           165   4242645.43
2024-01-03 00:00:00+03:00           169   4517004.59
2024-01-04 00:00:00+03:00           168   3624982.14
2024-01-05 00:00:00+03:00           181   4763831.59
2024-01-06 00:00:00+03:00           160   4450229.73
2024-01-07 00:00:00+03:00           158   3967384.29
2024-01-08 00:00:00+03:00           161   4201864.84
2024-01-09 00:00:00+03:00           175   4442862.12
2024-01-10 00:00:00+03:00           169   4249550.74

### Задание 2. Добавить к дневной таблице 7-дневное скользящее среднее по выручке

In [5]:
# Скользящее среднее выручки за 7 дней (min_periods=1 — считаем даже для первых дней)
daily_orders["revenue_rolling_7d"] = (
    daily_orders["revenue_sum"]
    .rolling(window=7, min_periods=1)
    .mean()
)

print("Таблица с 7-дневным скользящим средним (первые 15 дней):")
daily_orders[["revenue_sum", "revenue_rolling_7d"]].head(15)

Таблица с 7-дневным скользящим средним (первые 15 дней):


                           revenue_sum  revenue_rolling_7d
order_datetime_msk
2024-01-01 00:00:00+03:00   3888229.53        3.888230e+06
2024-01-02 00:00:00+03:00   4242645.43        4.065437e+06
2024-01-03 00:00:00+03:00   4517004.59        4.215960e+06
2024-01-04 00:00:00+03:00   3624982.14        4.068215e+06
2024-01-05 00:00:00+03:00   4763831.59        4.207339e+06
2024-01-06 00:00:00+03:00   4450229.73        4.247821e+06
2024-01-07 00:00:00+03:00   3967384.29        4.207758e+06
2024-01-08 00:00:00+03:00   4201864.84        4.252563e+06
2024-01-09 00:00:00+03:00   4442862.12        4.281166e+06
2024-01-10 00:00:00+03:00   4249550.74        4.242958e+06
2024-01-11 00:00:00+03:00   4415233.33        4.355851e+06
2024-01-12 00:00:00+03:00   3630431.00        4.193937e+06
2024-01-13 00:00:00+03:00   4310497.15        4.173975e+06
2024-01-14 00:00:00+03:00   5268443.08        4.359840e+06
2024-01-15 00:00:00+03:00   4395603.31        4.387517e+06

### Задание 3. Найти день с максимальным значением скользящего среднего

In [6]:
# День с наибольшим 7-дневным скользящим средним выручки
best_day = daily_orders["revenue_rolling_7d"].idxmax()

print("День с максимальным скользящим средним:", best_day.date())
print()
print(daily_orders.loc[best_day])

День с максимальным скользящим средним: 2024-02-28

orders_count          1.790000e+02
revenue_sum           4.514644e+06
revenue_rolling_7d    4.545486e+06
Name: 2024-02-28 00:00:00+03:00, dtype: float64
